# Monte-Carlo statistics of turbulent beam propagation

A single propagation through a random medium is one *realization*, and carries
no statistical meaning in isolation: the speckle pattern it produces is a
property of that particular draw of the medium rather than of the turbulence.
The quantities of physical interest are moments over the ensemble of
realizations.

This notebook states what those moments are, estimates them with
`ensemble_solve`, and then addresses the question that governs any Monte-Carlo
study: how many realizations are required. The answer is not a single number.
It depends on which statistic is being estimated, and for the scintillation
index it is considerably larger than is commonly assumed.

## 1. The equation, with a random index

The package integrates the slowly varying envelope $\psi$, defined by
$E = \psi e^{ikz}$ with $k = k_0 n_0$. The governing equation is

$$
\frac{\partial \psi}{\partial z}
  = \frac{i}{2 k_0 n_0} \nabla_\perp^2 \psi
  + i k_0 \, \delta n(x, y, z; \omega) \, \psi ,
$$

where $\delta n = n - n_0$ is the refractive index perturbation and $\omega$
labels one realization of the random medium. The first term is diffraction, the
second refraction. Everything that follows concerns the dependence on $\omega$.

## 2. Split-step propagation as a sequence of phase screens

Over one step $\Delta z$ the two operators are applied in sequence. Diffraction
is diagonal in Fourier space and refraction is diagonal in real space, so each
is exact in its own representation:

$$
\psi(z + \Delta z) = \underbrace{e^{\,i k_0 \delta n \Delta z}}_{\text{phase screen}}
  \; \mathcal{F}^{-1}\!\left[
    e^{\,-i |\mathbf{k}_\perp|^2 \Delta z / 2 k_0 n_0} \,
    \mathcal{F}[\psi]
  \right] + O(\Delta z^2).
$$

The refraction factor is a *thin random phase screen*. In this notebook the
screens are generated on demand by `phase_screen`, with the step index folded
into the PRNG key, so that each slice draws an independent screen and no index
volume is ever stored.

Independent screens make the medium $\delta$-correlated in $z$. This is the
standard thin-screen model and is appropriate when the step size already
exceeds the longitudinal correlation length of the physical medium, but it is
an approximation and not a discretization of a $z$-correlated volume. Where
longitudinal correlation matters, `random_medium_spectral` generates the
volume instead.

## 3. What an ensemble average is

Each PRNG key is one draw $\omega$ of the medium. Writing
$I(\mathbf{r}; \omega) = |\psi(\mathbf{r}; \omega)|^2$, the quantities of
interest are expectations over $\omega$:

| quantity | definition | meaning |
|---|---|---|
| long-exposure intensity | $\langle I \rangle(\mathbf{r})$ | what a time-integrating detector records |
| scintillation index | $\sigma_I^2 = \dfrac{\langle I^2 \rangle}{\langle I \rangle^2} - 1$ | normalized intensity fluctuation |
| beam wander | $\mathrm{Var}_\omega\!\left[ \dfrac{\int x I \, dA}{\int I \, dA} \right]$ | spread of the instantaneous centroid |
| degree of coherence | $\mu(\mathbf{r}) = \dfrac{\left| \langle \psi(\mathbf{r}_0) \psi^*(\mathbf{r}) \rangle \right|}{\sqrt{\langle I(\mathbf{r}_0) \rangle \langle I(\mathbf{r}) \rangle}}$ | field correlation about a reference point |

A Monte-Carlo estimate replaces $\langle \cdot \rangle$ by an average over $N$
independent realizations. The error of such an estimate decreases as
$N^{-1/2}$, but with a prefactor set by the variance of the quantity being
averaged. That prefactor is what distinguishes these statistics from one
another in practice, and section 8 measures it.

Note that $\sigma_I^2$ involves $\langle I^2 \rangle$. In developed speckle the
intensity is close to exponentially distributed, for which the fourth field
moment is large; the estimator therefore has a heavy tail. Beam wander, by
contrast, is a first moment of a bounded quantity.

In [ ]:
!pip install git+https://github.com/Forgotten/paraxial_wave_solver.git
!pip install matplotlib

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import paraxial_wave_solver as pws

# float32 is the JAX default and limits the attainable accuracy.
pws.enable_x64()

## 4. Configuration

A helium-neon beam at 632.8 nm propagating 2.5 m through water. The index
perturbation is specified per screen through its standard deviation; two
strengths are used, differing by a factor of 1.6, which turns out to separate
the two regimes of interest.

The transverse grid spacing is the quantity to watch. Stronger turbulence
produces finer speckle, and once the speckle is comparable to `dx` the
statistics describe the grid rather than the physics. Section 9 returns to
this.

In [ ]:
WAVELENGTH = 632.8e-9
REFRACTIVE_INDEX = 1.33
WAIST = 3e-3
CORRELATION_LENGTH = 2e-3

sim_config = pws.SimulationConfig(
    nx=128, ny=128, dx=2e-4, dy=2e-4, dz=2.5e-2, nz=100,
    wavelength=WAVELENGTH, n0=REFRACTIVE_INDEX,
)
solver_config = pws.SolverConfig(method='spectral', stepper='split_step')
pml_config = pws.PMLConfig(width_x=16, width_y=16, strength=2.0)

psi_0 = pws.gaussian_beam(sim_config, w0=WAIST)

REGIMES = {'weak': 5e-7, 'moderate': 8e-7}


def build_solver(strength):
    """Returns a solver whose `medium` argument is a single PRNG key."""
    def delta_n_fn(z, key):
        step = jnp.floor(z / sim_config.dz).astype(int)
        return pws.phase_screen(
            jax.random.fold_in(key, step), sim_config,
            CORRELATION_LENGTH, strength,
        )

    return pws.ParaxialWaveSolver(
        sim_config, solver_config, pml_config, delta_n_fn
    )


print(f"domain {sim_config.lx * 1e3:.1f} mm across, "
      f"{sim_config.lz:.2f} m long, dx = {sim_config.dx * 1e6:.0f} um")

## 5. One realization is not an answer

Three realizations of the same turbulence, differing only in their PRNG key.
The patterns are unrelated in detail; nothing in any single panel is a property
of the turbulence.

In [ ]:
solver = build_solver(REGIMES['moderate'])
three_keys = jax.random.split(jax.random.PRNGKey(0), 3)

single = jnp.stack([
    solver.solve(psi_0, medium=k, return_history=False)[0] for k in three_keys
])

extent_mm = [0, sim_config.lx * 1e3, 0, sim_config.ly * 1e3]
peak = float(jnp.max(jnp.abs(single) ** 2))

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for index, ax in enumerate(axes):
    ax.imshow(jnp.abs(single[index]).T ** 2, origin='lower', cmap='inferno',
              extent=extent_mm, vmin=0, vmax=peak)
    ax.set_title(f'realization {index + 1}')
    ax.set_xlabel('x (mm)')
axes[0].set_ylabel('y (mm)')
fig.suptitle('Three draws of the same medium', fontsize=12)
fig.tight_layout()

## 6. The ensemble

`ensemble_solve` maps the solver over a batch of keys. It applies `vmap` within
a chunk and `lax.map` across chunks, so the vectorization is retained while
peak memory is set by `chunk_size` rather than by the ensemble size. A plain
`vmap` over all 512 realizations would allocate 512 copies of every temporary
inside the step.

In [ ]:
N_REALIZATIONS = 512
CHUNK_SIZE = 64

fields = {}
for label, strength in REGIMES.items():
    keys = jax.random.split(jax.random.PRNGKey(2024), N_REALIZATIONS)
    fields[label], _ = pws.ensemble_solve(
        build_solver(strength), psi_0, keys, chunk_size=CHUNK_SIZE
    )
    print(f"{label:>9}: {fields[label].shape}")

## 7. Short exposure against long exposure

The ensemble mean is the visual content of an averaging measurement: a single
realization is speckled, while $\langle I \rangle$ is smooth and broader. The
broadening is not a loss of focus in any individual realization; it is the
record of beam wander displacing the instantaneous pattern between
realizations.

The third panel shows where the fluctuation is concentrated.

In [ ]:
# The scintillation index is normalized by the mean intensity, so in the dark
# background its denominator approaches zero and the ratio diverges. The map is
# therefore restricted to the illuminated region; unmasked, it would show only
# the grid edges.
for label in REGIMES:
    statistics = pws.ensemble_statistics(fields[label], sim_config)
    mean_intensity = statistics['mean_intensity']
    peak = float(jnp.max(mean_intensity))

    illuminated = mean_intensity > 0.01 * peak
    masked = jnp.where(illuminated, statistics['scintillation_index'], jnp.nan)
    upper = float(jnp.percentile(statistics['scintillation_index'][illuminated], 99))
    speckle = plt.get_cmap('viridis').copy()
    speckle.set_bad(color='0.92')

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
    panels = [
        (jnp.abs(fields[label][0]) ** 2, 'single realization', 'inferno',
         (0, peak)),
        (mean_intensity, r'ensemble mean $\langle I \rangle$', 'inferno',
         (0, peak)),
        (masked, r'$\sigma_I^2$ (illuminated region)', speckle, (0, upper)),
    ]
    for ax, (field, title, cmap, limits) in zip(axes, panels, strict=True):
        kwargs = {} if limits is None else {'vmin': limits[0], 'vmax': limits[1]}
        image = ax.imshow(jnp.asarray(field).T, origin='lower', cmap=cmap,
                          extent=extent_mm, **kwargs)
        ax.set_title(title)
        ax.set_xlabel('x (mm)')
        plt.colorbar(image, ax=ax, fraction=0.046)
    axes[0].set_ylabel('y (mm)')
    fig.suptitle(f'{N_REALIZATIONS} realizations, {label} turbulence',
                 fontsize=12)
    fig.tight_layout()

## 8. The statistics, with error bars

`ensemble_statistics` evaluates the quantities of section 3;
`ensemble_standard_error` bootstraps the realization axis to attach an
uncertainty to each. The bootstrap is the only reliable way to answer "is this
ensemble large enough", because the answer differs between statistics computed
from the very same ensemble.

In [ ]:
N_RESAMPLES = 200

statistics, errors = {}, {}
for label in REGIMES:
    statistics[label] = pws.ensemble_statistics(fields[label], sim_config)
    errors[label] = pws.ensemble_standard_error(
        fields[label],
        lambda ensemble: pws.ensemble_statistics(ensemble, sim_config),
        jax.random.PRNGKey(7),
        n_resamples=N_RESAMPLES,
    )

rows = [
    ('on-axis sigma_I^2', 'on_axis_scintillation_index', 1.0, ''),
    ('beam wander', 'wander_x', 1e6, 'um'),
    ('coherence radius', 'coherence_radius', 1e3, 'mm'),
    ('short-exposure width', 'short_exposure_width_x', 1e3, 'mm'),
    ('long-exposure width', 'long_exposure_width_x', 1e3, 'mm'),
]

print(f"{'quantity':<24}" + ''.join(f'{label:>26}' for label in REGIMES))
for title, key, scale, unit in rows:
    cells = ''
    for label in REGIMES:
        value = float(statistics[label][key]) * scale
        error = float(errors[label][key]) * scale
        cells += f'{value:14.3f} +/- {error:<6.3f}{unit:<4}'
    print(f"{title:<24}{cells}")

## 9. Spatial resolution is a precondition

Before any of the above is interpreted physically, the speckle it describes
must be resolved by the transverse grid. The relevant diagnostic is the
measured coherence radius expressed in units of `dx`: below roughly three
cells, the scintillation index is a discretization artefact and decreases under
grid refinement without converging.

This is the reason the two regimes above are separated by a factor of only 1.6
rather than by an order of magnitude. A saturated regime on this grid would
produce a coherence radius of one or two cells, and a scintillation index that
is a property of `dx`.

In [ ]:
print(f"{'regime':<12}{'r_coh (mm)':>13}{'r_coh / dx':>13}   resolved?")
for label in REGIMES:
    radius = float(statistics[label]['coherence_radius'])
    ratio = radius / sim_config.dx
    verdict = 'yes' if ratio > 3 else 'NO - refine the grid'
    print(f"{label:<12}{radius * 1e3:>13.3f}{ratio:>13.2f}   {verdict}")

## 10. How many realizations are required

The relative standard error of each statistic, against the number of
realizations used. Subsets of the ensemble already computed are reused, so this
costs only the bootstrap.

Both curves fall as $N^{-1/2}$, which appears as a straight line of slope
$-1/2$ on logarithmic axes. What differs is the offset: the scintillation index
sits roughly a factor of two to three above beam wander throughout, and since
the error falls as $N^{-1/2}$, closing a factor of $f$ in precision costs a
factor of $f^2$ in realizations.

In [ ]:
def on_axis_scintillation(ensemble):
    intensity = jnp.abs(ensemble) ** 2
    mean_intensity = jnp.mean(intensity, axis=0)
    index = jnp.unravel_index(jnp.argmax(mean_intensity), mean_intensity.shape)
    on_axis = intensity[:, index[0], index[1]]
    return jnp.mean(on_axis ** 2) / jnp.mean(on_axis) ** 2 - 1.0


def wander(ensemble):
    return pws.beam_wander(ensemble, sim_config)[0]


sizes = [16, 32, 64, 128, 256, 512]
curves = {}
for label in REGIMES:
    curves[label] = {}
    for name, statistic in (('sigma_I^2', on_axis_scintillation),
                            ('wander', wander)):
        relative = []
        for size in sizes:
            subset = fields[label][:size]
            value = float(statistic(subset))
            error = float(pws.ensemble_standard_error(
                subset, statistic, jax.random.PRNGKey(size),
                n_resamples=N_RESAMPLES,
            ))
            relative.append(100.0 * error / abs(value))
        curves[label][name] = relative

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), sharey=True)
for ax, label in zip(axes, REGIMES, strict=True):
    for name, style in (('sigma_I^2', 'o-'), ('wander', 's-')):
        ax.plot(sizes, curves[label][name], style, label=name, linewidth=1.8)
    reference = curves[label]['wander'][0] * np.sqrt(sizes[0] / np.array(sizes))
    ax.plot(sizes, reference, 'k--', alpha=0.5, linewidth=1.2,
            label=r'$N^{-1/2}$')
    ax.set_xscale('log', base=2)
    ax.set_yscale('log')
    ax.set_xlabel('realizations $N$')
    ax.set_title(f'{label} turbulence')
    ax.grid(True, alpha=0.3, which='both')
axes[0].set_ylabel('relative standard error (%)')
axes[0].legend()
fig.tight_layout()

print(f"{'N':>6}" + ''.join(f"{label + ' ' + n:>22}"
                            for label in REGIMES for n in ('sig', 'wan')))
for row, size in enumerate(sizes):
    cells = ''.join(f"{curves[label][n][row]:>22.1f}"
                    for label in REGIMES for n in ('sigma_I^2', 'wander'))
    print(f"{size:>6}{cells}")

## 11. Conclusions

1. **An ensemble average is the measurement.** A single realization is a draw,
   not a result. The long-exposure intensity is broader than any individual
   realization, and that broadening is the record of beam wander.

2. **The cost of a statistic is a property of the statistic.** Estimated from
   the same ensemble, beam wander and the scintillation index differ by a
   factor of two to three in relative precision. Since the error falls as
   $N^{-1/2}$, matching them costs four to nine times as many realizations.

3. **The scintillation index is also biased at small $N$.** It is a ratio of
   sample moments of a heavy-tailed quantity, and a small ensemble
   underestimates it. A short run therefore reports a value that is wrong
   rather than one that is visibly noisy, and the standard error alone does not
   reveal this.

4. **Resolution precedes statistics.** The coherence radius must exceed a few
   grid cells before a scintillation index means anything. A saturated regime
   demands a finer transverse grid, not merely a larger ensemble.

### Limitations

The thin-screen model used here is $\delta$-correlated in $z$: consecutive
screens are independent, so the longitudinal correlation of a physical medium
is not reproduced. For a comparison against weak-fluctuation (Rytov) theory,
which assumes a Kolmogorov inertial range, use `random_medium_spectral` with a
Von Karman spectrum; that generator materializes the index volume, so the
memory cost per realization rises from one transverse plane to $n_z$ of them
and the practical ensemble size falls accordingly.